In [9]:
# import libraries
import osmnx as ox
import pandas as pd
import geopandas as gpd

In [10]:
# Pull language schools data from OSM 
place_name = "Berlin, Germany"
tags = {"amenity": "language_school"}

gdf = ox.features_from_place(place_name, tags)

print(f"Total features found: {len(gdf)}")
print(f"\nAll available columns:")
print(gdf.columns.tolist())

Total features found: 55

All available columns:
['geometry', 'addr:city', 'addr:country', 'addr:housename', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb', 'amenity', 'name', 'website', 'contact:website', 'education', 'language:de', 'language:en', 'opening_hours', 'operator', 'description', 'language:es', 'phone', 'toilets:wheelchair', 'wheelchair', 'check_date', 'contact:email', 'contact:phone', 'language:ar', 'language:fr', 'language:it', 'language:no', 'language:pl', 'language:pt', 'language:ru', 'language:se', 'check_date:opening_hours', 'education:languages', 'email', 'opening_hours:signed', 'fax', 'source:name', 'source_ref', 'start_date', 'wikidata', 'wikipedia', 'opening_hours:url', 'smoking', 'wheelchair:description', 'level', 'brand', 'brand:wikidata', 'short_name', 'name:ar', 'name:en', 'name:ja', 'max_age', 'language:ja', 'operator:type', 'contact:fax', 'source', 'language:nn', 'tourism', 'language:tr', 'phone:mobile', 'language:da', 'language:el', 'lang

In [11]:
# Select relevant columns
columns_to_keep = [
    'geometry',
    'name',
    'addr:street',
    'addr:housenumber',
    'addr:postcode',
    'addr:city',
    'addr:suburb',
    'website',
    'contact:website',
    'phone',
    'contact:phone',
    'email',
    'contact:email',
    'operator',
    'operator:type',
    'opening_hours',
    'description',
    'education:languages',
    'language:de',
    'language:en',
    'language:fr',
    'language:es',
    'language:ar',
    'language:it',
    'language:ru',
    'language:tr',
    'language:zh',
    'wheelchair',
    'start_date',
    'wikidata',
    'short_name',
    'brand',
]

existing_cols = [c for c in columns_to_keep if c in gdf.columns]
gdf_selected = gdf[existing_cols].copy()
print(f"Selected {len(existing_cols)} columns")
gdf_selected.head(3)

Selected 32 columns


geometry                name  \
element id                                                          
node    486761746    POINT (13.40855 52.5371)    GLS Sprachschule   
        835553966   POINT (13.36021 52.48988)  BSI Sprachenschule   
        1192599547  POINT (13.40094 52.54821)   Mi Escuela Berlin   

                         addr:street addr:housenumber addr:postcode addr:city  \
element id                                                                      
node    486761746     Kastanienallee               82         10435    Berlin   
        835553966        Hauptstraße              159         10827    Berlin   
        1192599547  Schwedter Straße               81         10437    Berlin   

                        addr:suburb  \
element id                            
node    486761746   Prenzlauer Berg   
        835553966        Schöneberg   
        1192599547  Prenzlauer Berg   

                                                        website  \
element id                                                        
node    486761746   https://www.gls-berlin.de/sprachschule.html   
        835553966                                           NaN   
        1192599547            https://www.mi-escuela-berlin.de/   

                               contact:website            phone  ...  \
element id                                                       ...   
node    486761746                          NaN              NaN  ...   
        835553966   https://www.bsiberlin.com/              NaN  ...   
        1192599547                         NaN  +49 30 67949588  ...   

                   language:ar language:it language:ru language:tr  \
element id                                                           
node    486761746          NaN         NaN         NaN         NaN   
        835553966          NaN         NaN         NaN         NaN   
        1192599547         NaN         NaN         NaN         NaN   

                   language:zh wheelchair start_date wikidata short_name brand  
element id                                                                      
node    486761746          NaN        NaN        NaN      NaN        NaN   NaN  
        835553966          NaN        NaN        NaN      NaN        NaN   NaN  
        1192599547         NaN         no        NaN      NaN        NaN   NaN  

[3 rows x 32 columns]

In [12]:
# Load Berlin districts geojson
berlin_districts_gdf = gpd.read_file("../sources/lor_ortsteile.geojson")

# Keep only needed columns 
berlin_districts_cols = berlin_districts_gdf[['OTEIL', 'BEZIRK', 'spatial_name', 'geometry']].copy()

berlin_districts_cols['spatial_name'] = berlin_districts_cols['spatial_name'].astype(str)

print(f"Districts loaded: {len(berlin_districts_gdf)}")
print(f"Sample spatial_name values: {berlin_districts_cols['spatial_name'].head(5).tolist()}")

Districts loaded: 96
Sample spatial_name values: ['0101', '0102', '0103', '0104', '0105']


In [13]:
# Use same CRS
gdf_selected = gdf_selected.to_crs("EPSG:4326")
berlin_districts_cols = berlin_districts_cols.to_crs("EPSG:4326")


gdf_selected['geometry'] = gdf_selected['geometry'].apply(
    lambda geom: geom.centroid if geom.geom_type != 'Point' else geom
)

# Spatial join 
gdf_joined = gpd.sjoin(
    gdf_selected,
    berlin_districts_cols,
    how='left',
    predicate='within'
)

print(f"Columns after join: {gdf_joined.columns.tolist()}")
print(f"\nspatial_name sample: {gdf_joined['spatial_name'].head(5).tolist()}")

Columns after join: ['geometry', 'name', 'addr:street', 'addr:housenumber', 'addr:postcode', 'addr:city', 'addr:suburb', 'website', 'contact:website', 'phone', 'contact:phone', 'email', 'contact:email', 'operator', 'operator:type', 'opening_hours', 'description', 'education:languages', 'language:de', 'language:en', 'language:fr', 'language:es', 'language:ar', 'language:it', 'language:ru', 'language:tr', 'language:zh', 'wheelchair', 'start_date', 'wikidata', 'short_name', 'brand', 'index_right', 'OTEIL', 'BEZIRK', 'spatial_name']

spatial_name sample: ['0301', '0701', '0301', '0301', '0501']


In [14]:
# Rename columns 
gdf_joined = gdf_joined.rename(columns={
    'BEZIRK': 'district',
    'OTEIL': 'neighborhood',
    'spatial_name': 'neighborhood_id'
})

# Apply district_id mapping 
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

gdf_joined['district_id'] = gdf_joined['district'].map(district_mapping).astype(str)

print(f"Schools mapped: {gdf_joined['district'].notna().sum()}/{len(gdf_joined)}")
print(f"neighborhood_id sample: {gdf_joined['neighborhood_id'].head(5).tolist()}")
print(f"\nDistrict distribution:")
print(gdf_joined['district'].value_counts())

Schools mapped: 55/55
neighborhood_id sample: ['0301', '0701', '0301', '0301', '0501']

District distribution:
district
Pankow                        11
Friedrichshain-Kreuzberg       9
Charlottenburg-Wilmersdorf     8
Tempelhof-Schöneberg           7
Mitte                          7
Neukölln                       6
Reinickendorf                  2
Steglitz-Zehlendorf            2
Spandau                        1
Treptow-Köpenick               1
Lichtenberg                    1
Name: count, dtype: int64


In [16]:
# Final cleanup
gdf_final = gdf_joined.reset_index()

# Extract id
gdf_final['id'] = gdf_final['id'].astype(str)

# Extract lat/lon from geometry
gdf_final['latitude'] = gdf_final['geometry'].apply(lambda p: round(p.y, 6))
gdf_final['longitude'] = gdf_final['geometry'].apply(lambda p: round(p.x, 6))


gdf_final['geometry_str'] = gdf_final['geometry'].apply(
    lambda p: f"POINT({round(p.x, 6)} {round(p.y, 6)})"
)
gdf_final = gdf_final.drop(columns=['geometry'])
gdf_final = gdf_final.rename(columns={'geometry_str': 'geometry'})


if 'contact:website' in gdf_final.columns:
    gdf_final['website'] = gdf_final['website'].fillna(gdf_final['contact:website'])
if 'contact:phone' in gdf_final.columns:
    gdf_final['phone'] = gdf_final['phone'].fillna(gdf_final['contact:phone'])
if 'contact:email' in gdf_final.columns:
    gdf_final['email'] = gdf_final['email'].fillna(gdf_final['contact:email'])

# Drop irrelevant columns
gdf_final = gdf_final.drop(
    columns=['element', 'amenity', 'contact:website', 'contact:phone',
             'contact:email', 'index_right'],
    errors='ignore'
)

print(f"neighborhood_id sample: {gdf_final['neighborhood_id'].head(5).tolist()}")
print(f"Final shape: {gdf_final.shape}")
gdf_final.head(3)

neighborhood_id sample: ['0301', '0701', '0301', '0301', '0501']
Final shape: (55, 36)


,id,name,addr:street,addr:housenumber,addr:postcode,addr:city,addr:suburb,website,phone,email,...,wikidata,short_name,brand,neighborhood,district,neighborhood_id,district_id,latitude,longitude,geometry
0,486761746,GLS Sprachschule,Kastanienallee,82,10435,Berlin,Prenzlauer Berg,https://www.gls-berlin.de/sprachschule.html,NaN,NaN,...,NaN,NaN,NaN,Prenzlauer Berg,Pankow,0301,11003003,52.537096,13.408550,POINT(13.40855 52.537096)
1,835553966,BSI Sprachenschule,Hauptstraße,159,10827,Berlin,Schöneberg,https://www.bsiberlin.com/,NaN,NaN,...,NaN,NaN,NaN,Schöneberg,Tempelhof-Schöneberg,0701,11007007,52.489876,13.360206,POINT(13.360206 52.489876)
2,1192599547,Mi Escuela Berlin,Schwedter Straße,81,10437,Berlin,Prenzlauer Berg,https://www.mi-escuela-berlin.de/,+49 30 67949588,NaN,...,NaN,NaN,NaN,Prenzlauer Berg,Pankow,0301,11003003,52.548212,13.400945,POINT(13.400945 52.548212)


In [21]:
# Save transformed dataset
output_path = 'data/language_schools_transformed.csv'
gdf_final.to_csv(output_path, index=False)
print(f"Saved {len(gdf_final)} schools to {output_path}")

# Check
df_check = pd.read_csv(output_path, dtype={'neighborhood_id': str, 'district_id': str})
print(f"\nSchools per district:")
print(df_check['district'].value_counts())

Saved 55 schools to data/language_schools_transformed.csv

Schools per district:
district
Pankow                        11
Friedrichshain-Kreuzberg       9
Charlottenburg-Wilmersdorf     8
Tempelhof-Schöneberg           7
Mitte                          7
Neukölln                       6
Reinickendorf                  2
Steglitz-Zehlendorf            2
Spandau                        1
Treptow-Köpenick               1
Lichtenberg                    1
Name: count, dtype: int64
